# NeuraRoads - 01 Data Exploration

Explore the 10-class YOLO dataset: split sizes, class distribution and sample annotated images. Use the project `venv` kernel.

In [ ]:
import sys, os
from pathlib import Path
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
SRC = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
from utils.config_loader import load_config, PROJECT_ROOT
cfg = load_config('model_config')
names = {int(k): v for k, v in cfg['detector']['class_names'].items()}
print('classes:', names)

In [ ]:
from collections import Counter
root = PROJECT_ROOT / 'data' / 'datasets'
counts = Counter()
for split in ['train', 'val', 'test']:
    idir, ldir = root / 'images' / split, root / 'labels' / split
    n_img = len(list(idir.glob('*.jpg'))) if idir.is_dir() else 0
    n_lbl = len(list(ldir.glob('*.txt'))) if ldir.is_dir() else 0
    print(f'{split}: images={n_img} labels={n_lbl}')
    if ldir.is_dir():
        for f in ldir.glob('*.txt'):
            for line in f.read_text().splitlines():
                p = line.split()
                if len(p) == 5:
                    counts[names.get(int(float(p[0])), p[0])] += 1
counts

In [ ]:
import matplotlib.pyplot as plt
labels = [names[i] for i in sorted(names)]
vals = [counts.get(l, 0) for l in labels]
plt.figure(figsize=(10, 4))
plt.bar(labels, vals)
plt.yscale('log'); plt.ylabel('instances (log)')
plt.title('NeuraRoads class distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

In [ ]:
import cv2, random
imgs = list((root / 'images' / 'train').glob('*.jpg'))
random.seed(0)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, ip in zip(axes, random.sample(imgs, 3)):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in (root / 'labels' / 'train' / (ip.stem + '.txt')).read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = float(cx)*w, float(cy)*h, float(bw)*w, float(bh)*h
        x1, y1 = int(cx-bw/2), int(cy-bh/2)
        cv2.rectangle(img, (x1, y1), (int(cx+bw/2), int(cy+bh/2)), (0,255,0), 2)
        cv2.putText(img, names[int(c)], (x1, max(12,y1-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()